# Module B2 : Qiskit — module `circuit`

Ce notebook accompagne le module A5.  
On utilise le module `QuantumCircuit` de Qiskit pour construire et visualiser des circuits quantiques.

**Plan :**
1. Construction de circuits quantiques
2. Portes logiques et leurs matrices
3. Construction d'une paire de Bell
4. Mesures dans les circuits

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator

---
## 1. Construction de circuits quantiques

Un **circuit quantique** est une séquence de portes logiques appliquées à des qubits.
Tout circuit part de l'état initial $\ket{00\cdots0}$.

### 1.1 Créer un circuit et y ajouter des portes

In [ ]:
# Créer un circuit à 1 qubit
circuit = QuantumCircuit(1)

# Ajouter la porte de Hadamard sur le qubit 0
circuit.h(0)

# Visualiser le circuit
circuit.draw('mpl')

In [ ]:
# Calculer l'état final du circuit
etat_final = Statevector.from_instruction(circuit)
print("État final après H|0> :")
display(etat_final.draw('latex'))

### 1.2 Enchaîner des portes

L'**ordre** des portes est important : en général, la multiplication
de matrices n'est **pas commutative**.

In [ ]:
# H puis X
circuit_hx = QuantumCircuit(1)
circuit_hx.h(0)
circuit_hx.x(0)

etat_hx = Statevector.from_instruction(circuit_hx)
print("H puis X sur |0> :")
display(etat_hx.draw('latex'))
circuit_hx.draw('mpl')

In [ ]:
# X puis H
circuit_xh = QuantumCircuit(1)
circuit_xh.x(0)
circuit_xh.h(0)

etat_xh = Statevector.from_instruction(circuit_xh)
print("X puis H sur |0> :")
display(etat_xh.draw('latex'))
circuit_xh.draw('mpl')

---
## 2. Portes logiques et leurs matrices

Chaque porte du circuit correspond à une matrice unitaire.
On peut extraire la matrice d'un circuit avec `Operator`.

### 2.1 Portes à un qubit : X, Z, H

In [ ]:
# Matrice derrière la porte X
qc = QuantumCircuit(1)
qc.x(0)
print("Matrice X :")
display(Operator(qc).draw('latex'))
qc.draw('mpl')

In [ ]:
# Matrice derrière la porte Z
qc = QuantumCircuit(1)
qc.z(0)
print("Matrice Z :")
display(Operator(qc).draw('latex'))
qc.draw('mpl')

In [ ]:
# Matrice derrière la porte H
qc = QuantumCircuit(1)
qc.h(0)
print("Matrice H :")
display(Operator(qc).draw('latex'))
qc.draw('mpl')

### 2.2 Portes à deux qubits : SWAP, CNOT, CZ

In [ ]:
# Porte SWAP
qc = QuantumCircuit(2)
qc.swap(0, 1)
print("Matrice SWAP :")
display(Operator(qc).draw('latex'))
qc.draw('mpl')

In [ ]:
# SWAP sur |01> → doit donner |10>
qc = QuantumCircuit(2)
qc.initialize([0, 1], 0)   # qubit 0 = |1>
qc.initialize([1, 0], 1)   # qubit 1 = |0>
qc.swap(0, 1)

etat = Statevector.from_instruction(qc)
print("SWAP|01> :")
display(etat.draw('latex'))
qc.draw('mpl')

In [ ]:
# Porte CNOT (CX)
qc = QuantumCircuit(2)
qc.cx(0, 1)  # qubit 0 = contrôle, qubit 1 = cible
print("Matrice CNOT :")
display(Operator(qc).draw('latex'))
qc.draw('mpl')

In [ ]:
# CNOT sur |10> → doit donner |11>
qc = QuantumCircuit(2)
qc.initialize([1, 0], 0)   # qubit 0 = |0>
qc.initialize([0, 1], 1)   # qubit 1 = |1>
qc.cx(1, 0)  # qubit 1 contrôle, qubit 0 cible

etat = Statevector.from_instruction(qc)
print("CNOT|10> :")
display(etat.draw('latex'))
qc.draw('mpl')

In [ ]:
# Porte CZ
qc = QuantumCircuit(2)
qc.cz(0, 1)
print("Matrice CZ :")
display(Operator(qc).draw('latex'))
qc.draw('mpl')

In [ ]:
# CZ sur |11> → doit donner -|11>
qc = QuantumCircuit(2)
qc.initialize([0, 1], 0)   # qubit 0 = |1>
qc.initialize([0, 1], 1)   # qubit 1 = |1>
qc.cz(0, 1)

etat = Statevector.from_instruction(qc)
print("CZ|11> :")
display(etat.draw('latex'))
qc.draw('mpl')

---
## 3. Construction d'une paire de Bell

On crée l'état de Bell $\ket{\Phi^+} = \frac{1}{\sqrt{2}}(\ket{00} + \ket{11})$
en appliquant une porte de Hadamard puis une porte CNOT.

In [ ]:
# Circuit de paire de Bell
bell_circuit = QuantumCircuit(2)
bell_circuit.h(0)       # Hadamard sur qubit 0
bell_circuit.cx(0, 1)   # CNOT : qubit 0 contrôle, qubit 1 cible

# Visualiser
bell_circuit.draw('mpl')

In [ ]:
# Vérifier l'état produit
etat_bell = Statevector.from_instruction(bell_circuit)
print("État de Bell Φ+ :")
display(etat_bell.draw('latex'))

In [ ]:
# Vérifier que c'est bien intriqué (décomposition de Schmidt)
from qiskit.quantum_info import schmidt_decomposition
from display_utils import my_display_of_schmidt_decomposition

res = schmidt_decomposition(etat_bell, [0])
print(f"Rang de Schmidt : {len(res)}  → intriqué!")
my_display_of_schmidt_decomposition(res)

---
## 4. Mesures dans les circuits

On ajoute une opération de mesure au circuit.
Le résultat est stocké dans un **bit classique**.

In [ ]:
# Circuit H + mesure
qc = QuantumCircuit(1, 1)  # 1 qubit, 1 bit classique
qc.h(0)
qc.measure(0, 0)  # mesurer le qubit 0 → stocker dans le bit classique 0

qc.draw('mpl')

In [ ]:
# Mesure sur la paire de Bell
bell_measure = QuantumCircuit(2, 2)
bell_measure.h(0)
bell_measure.cx(0, 1)
bell_measure.measure([0, 1], [0, 1])

bell_measure.draw('mpl')

In [ ]:
# Statistiques de mesure sur la paire de Bell
etat_bell = Statevector.from_instruction(bell_circuit)  # circuit sans mesure
counts = etat_bell.sample_counts(1000)
print("Résultats sur 1000 mesures :")
print(counts)

from qiskit.visualization import plot_histogram
plot_histogram(counts)

On observe uniquement $\ket{00}$ et $\ket{11}$ — les résultats des deux qubits
sont toujours parfaitement corrélés, comme prévu par l'intrication.

### Exercices

<mark>**Exercice B2.1**</mark> : Construire un circuit qui crée l'état de Bell
$\ket{\Psi^+} = \frac{1}{\sqrt{2}}(\ket{01} + \ket{10})$.

*Indice : partir de l'état $\ket{10}$ en appliquant X sur le qubit 1 avant H + CNOT.*

<mark>**Exercice B2.2**</mark> : Construire un circuit à 2 qubits qui applique
Hadamard en parallèle sur les deux qubits ($H \otimes H$ partant de $\ket{00}$).
Vérifier que l'état final est une superposition uniforme sur 4 états.